2 序列模型

2.1 理论计算题

字符序列: ababc
词汇表: ['a', 'b', 'c']
转移计数:
  a->b: 2
  b->a: 1
  b->c: 1

拉普拉斯平滑（加1平滑）:
公式: p(x_t|x_{t-1}) = (count(x_{t-1}, x_t) + 1) / (count(x_{t-1}) + |V|)
词汇表大小 |V| = 3

1. p('a'|'b') = (1 + 1) / (2 + 3) = 0.4000
2. p('c'|'b') = (1 + 1) / (2 + 3) = 0.4000

验证: p('a'|'b') + p('b'|'b') + p('c'|'b') = 1.0000

2.2 文本预处理函数preprocess_text(text, n)
text: 输入文本字符串
n: 滑动窗口大小（特征序列长度）
Returns:
    vocab_dict: 词汇表字典 {词: ID}
    features: 特征列表，每个特征是长度为n的词列表
    labels: 标签列表，每个标签是下一个词
    

In [7]:
import re
from collections import Counter

def preprocess_text(text, n):
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text_lower = text.lower()
    text_clean = re.sub(r'[^a-z\s]', '', text_lower)
    
    # 2. 按空格分词
    tokens = text_clean.split()
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(tokens)
    sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
    vocab_dict = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成特征序列和标签
    features = []
    labels = []
    
    for i in range(len(tokens) - n):
        # 特征：当前窗口的n个词
        feature = tokens[i:i+n]
        # 标签：窗口后的下一个词
        label = tokens[i+n]
        features.append(feature)
        labels.append(label)
    
    return vocab_dict, features, labels

# 测试
text = "The time machine"
n = 2
vocab_dict, features, labels = preprocess_text(text, n)

print(f"输入文本: '{text}'")
print(f"n = {n}")
print(f"词汇表: {vocab_dict}")
print(f"特征列表: {features}")
print(f"标签列表: {labels}")

# 额外测试
print("\n额外测试:")
text2 = "This is a DeepLearning test!"
n = 3
vocab_dict2, features2, labels2 = preprocess_text(text2, n)
print(f"文本: '{text2}'")
print(f"词汇表: {vocab_dict2}")
print(f"特征: {features2}")
print(f"标签: {labels2}")

输入文本: 'The time machine'
n = 2
词汇表: {'the': 0, 'time': 1, 'machine': 2}
特征列表: [['the', 'time']]
标签列表: ['machine']

额外测试:
文本: 'This is a DeepLearning test!'
词汇表: {'this': 0, 'is': 1, 'a': 2, 'deeplearning': 3, 'test': 4}
特征: [['this', 'is', 'a'], ['is', 'a', 'deeplearning']]
标签: ['deeplearning', 'test']


3 循环神经网络

3.1 理论计算题

显现RNN（无偏置）
定义：h_t = W_hh * h_{t-1} + W_hx * x_t
输出：o_t = W_oh * h_t
损失函数为平分损失：L = (1/2) * Σ_{t=1}^{T} (o_t - y_t)^2

梯度推导:
∂L/∂W_hh = Σ_{t=1}^{T} ∂L/∂o_t * ∂o_t/∂h_t * ∂h_t/∂W_hh

其中 ∂L/∂o_t = o_t - y_t = δ_t
∂o_t/∂h_t = W_oh^T

∂h_t/∂W_hh = h_{t-1} + W_hh * ∂h_{t-1}/∂W_hh

展开到所有时间步:
∂L/∂W_hh = Σ_{t=1}^{T} (∂L/∂h_t) * h_{t-1}^T
其中 ∂L/∂h_t = W_oh^T * δ_t + W_hh^T * ∂L/∂h_{t+1}

梯度消失或爆炸的条件:
1. 梯度爆炸: ||W_hh|| > 1 时，随着时间步增加，(W_hh)^{t-k} 呈指数增长
2. 梯度消失: ||W_hh|| < 1 时，随着时间步增加，(W_hh)^{t-k} 呈指数衰减

具体条件:
- 如果 W_hh 的最大特征值 > 1，梯度爆炸
- 如果 W_hh 的最大特征值 < 1，梯度消失
- 如果 W_hh 的最大特征值 = 1，梯度稳定

3.2 RNN单元前向传播和单步反向传播

In [8]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hh, W_hx, b_h):
    """
    Args:
        x_t: 输入 (batch_size, input_size)
        h_prev: 上一隐藏状态 (batch_size, hidden_size)
        W_hh: 隐藏状态权重 (hidden_size, hidden_size)
        W_hx: 输入权重 (hidden_size, input_size)
        b_h: 偏置 (hidden_size,)
    Returns:
        h_t: 当前隐藏状态 (batch_size, hidden_size)
        cache: 缓存用于反向传播
    """
    # 计算当前隐藏状态
    h_t = np.tanh(np.dot(h_prev, W_hh) + np.dot(x_t, W_hx.T) + b_h)
    
    # 缓存中间变量
    cache = (x_t, h_prev, W_hh, W_hx, b_h, h_t)
    
    return h_t, cache

def rnn_cell_backward(dh_next, cache):
    """
    RNN单元单步反向传播
    Args:
        dh_next: 损失对h_t的梯度 (batch_size, hidden_size)
        cache: 前向传播缓存的元组
    
    Returns:
        dx_t: 损失对x_t的梯度 (batch_size, input_size)
        dh_prev: 损失对h_prev的梯度 (batch_size, hidden_size)
        dW_hh: 损失对W_hh的梯度 (hidden_size, hidden_size)
        dW_hx: 损失对W_hx的梯度 (hidden_size, input_size)
        db_h: 损失对b_h的梯度 (hidden_size,)
    """
    x_t, h_prev, W_hh, W_hx, b_h, h_t = cache
    
    # tanh的导数: dtanh = 1 - tanh^2
    dtanh = 1 - h_t ** 2
    
    # dh_next 通过 tanh 的导数
    dh = dh_next * dtanh
    
    # 计算各梯度
    # 对输入的梯度
    dx_t = np.dot(dh, W_hx)
    
    # 对前一隐藏状态的梯度
    dh_prev = np.dot(dh, W_hh.T)
    
    # 对权重矩阵的梯度
    dW_hh = np.dot(h_prev.T, dh)  # (hidden_size, batch_size) * (batch_size, hidden_size)
    dW_hx = np.dot(dh.T, x_t)     # (hidden_size, batch_size) * (batch_size, input_size)
    
    # 对偏置的梯度
    db_h = np.sum(dh, axis=0)     # (hidden_size,)
    
    return dx_t, dh_prev, dW_hh, dW_hx, db_h

# 测试
print("测试RNN单元前向和反向传播:")

# 设置参数
batch_size = 2
input_size = 3
hidden_size = 4

# 随机初始化
np.random.seed(42)
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hh = np.random.randn(hidden_size, hidden_size)
W_hx = np.random.randn(hidden_size, input_size)
b_h = np.random.randn(hidden_size)

print(f"batch_size = {batch_size}, input_size = {input_size}, hidden_size = {hidden_size}")

# 前向传播
h_t, cache = rnn_cell_forward(x_t, h_prev, W_hh, W_hx, b_h)
print(f"\n前向传播结果:")
print(f"h_t 形状: {h_t.shape}")
print(f"h_t 前5个值: {h_t.flatten()[:5]}")

# 反向传播
dh_next = np.random.randn(batch_size, hidden_size)
dx_t, dh_prev, dW_hh, dW_hx, db_h = rnn_cell_backward(dh_next, cache)

print(f"\n反向传播梯度:")
print(f"dx_t 形状: {dx_t.shape}")
print(f"dh_prev 形状: {dh_prev.shape}")
print(f"dW_hh 形状: {dW_hh.shape}")
print(f"dW_hx 形状: {dW_hx.shape}")
print(f"db_h 形状: {db_h.shape}")

# 数值梯度验证（近似）
print("\n数值梯度验证（近似检查）:")
epsilon = 1e-6

# 对W_hh进行数值梯度检查
W_hh_flat = W_hh.flatten()
dW_hh_flat = dW_hh.flatten()
dW_hh_num = np.zeros_like(W_hh_flat)

for i in range(min(5, len(W_hh_flat))):  # 只检查前5个
    W_hh_plus = W_hh.copy()
    W_hh_plus.flat[i] += epsilon
    h_t_plus, _ = rnn_cell_forward(x_t, h_prev, W_hh_plus, W_hx, b_h)
    
    W_hh_minus = W_hh.copy()
    W_hh_minus.flat[i] -= epsilon
    h_t_minus, _ = rnn_cell_forward(x_t, h_prev, W_hh_minus, W_hx, b_h)
    
    # 假设损失函数 L = sum(h_t) 用于近似
    loss_plus = np.sum(h_t_plus)
    loss_minus = np.sum(h_t_minus)
    dW_hh_num[i] = (loss_plus - loss_minus) / (2 * epsilon)

print(f"数值梯度 (前5个): {dW_hh_num[:5]}")
print(f"解析梯度 (前5个): {dW_hh_flat[:5]}")
print(f"相对误差: {np.linalg.norm(dW_hh_num[:5] - dW_hh_flat[:5]) / (np.linalg.norm(dW_hh_num[:5]) + 1e-8):.6f}")

测试RNN单元前向和反向传播:
batch_size = 2, input_size = 3, hidden_size = 4

前向传播结果:
h_t 形状: (2, 4)
h_t 前5个值: [-0.99985222 -0.99305969 -0.98685334 -0.46252019  0.96245027]

反向传播梯度:
dx_t 形状: (2, 3)
dh_prev 形状: (2, 4)
dW_hh 形状: (4, 4)
dW_hx 形状: (4, 3)
db_h 形状: (4,)

数值梯度验证（近似检查）:
数值梯度 (前5个): [-0.03368228 -0.01324568 -0.35111065  0.78158123 -0.03409257]
解析梯度 (前5个): [-0.01128214  0.03660476  0.27977161 -2.46985041 -0.01122684]
相对误差: 3.859645


4 高级循环神经网络

4.1 深度双向RNN参数计算:
 L: 层数
 H: 每层隐藏单元数
 D: 输入维度
 O: 输出维度（仅最后输出层）

每层双向RNN包含:
1. 前向RNN层:
    输入权重 W_hx: H × (输入维度)
    隐藏权重 W_hh: H × H
    偏置 b_h: H
    输出权重 W_oh: O × H (仅最后一层)
    输出偏置 b_o: O (仅最后一层)

2. 后向RNN层:
    输入权重 W_hx: H × (输入维度)
    隐藏权重 W_hh: H × H
    偏置 b_h: H
    输出权重 W_oh: O × H (仅最后一层)
    输出偏置 b_o: O (仅最后一层)

参数计算:

对于第1层:
 前向: H*D + H*H + H = H(D + H + 1)
 后向: H*D + H*H + H = H(D + H + 1)
 小计: 2H(D + H + 1)

对于第 l 层 (2 ≤ l ≤ L):
 输入维度为 2H (来自前一层的双向拼接)
 前向: H*(2H) + H*H + H = H(3H + 1)
 后向: H*(2H) + H*H + H = H(3H + 1)
 小计: 2H(3H + 1)

对于输出层 (仅在最后一层):
 前向输出权重: O*H
 前向输出偏置: O
 后向输出权重: O*H
 后向输出偏置: O
 小计: 2O(H + 1)

简化:
总参数 = 2H(D + H + 1 + (L-1)(3H + 1)) + 2O(H + 1)
= 2H(D + H + 1 + 3LH - 3H + L - 1) + 2O(H + 1)
= 2H(D + 3LH - 2H + L) + 2O(H + 1)
= 2H(D + L(3H + 1) - 2H) + 2O(H + 1)

总参数 = 2H(D + H + 1) + (L-1)*2H(3H + 1) + 2O(H + 1)

4.2 双向RNN编码器

In [9]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super(BidirectionalRNNEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # RNN实现双向
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=False,  # (seq_len, batch, input_dim)
            bidirectional=True
        )
    
    def forward(self, X):
        """
        前向传播
        Args:
            X: 输入序列 (seq_len, batch, input_dim)
        
        Returns:
            outputs: 每个时间步拼接的隐藏状态 (seq_len, batch, 2*hidden_dim)
            final_hidden: 最终时间步拼接的隐藏状态 (batch, 2*hidden_dim)
        """
        # RNN前向传播
        # output: (seq_len, batch, 2*hidden_dim)
        # h_n: (2*num_layers, batch, hidden_dim)
        output, h_n = self.rnn(X)
        
        # 获取前向和后向的最终隐藏状态
        # h_n[-2] 是最后一层前向的最终状态
        # h_n[-1] 是最后一层后向的最终状态
        forward_last = h_n[-2, :, :]  
        backward_last = h_n[-1, :, :]
        
        # 拼接前向和后向的最终隐藏状态
        final_hidden = torch.cat([forward_last, backward_last], dim=1)  # (batch, 2*hidden_dim)
        
        return output, final_hidden

# 手动实现的双向RNN
class BidirectionalRNNManual(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(BidirectionalRNNManual, self).__init__()
        self.hidden_dim = hidden_dim
        
        # 前向RNN
        self.rnn_forward = nn.RNNCell(input_dim, hidden_dim)
        # 后向RNN
        self.rnn_backward = nn.RNNCell(input_dim, hidden_dim)
    
    def forward(self, X):
        seq_len, batch_size, input_dim = X.shape
        
        # 前向传播
        h_forward = torch.zeros(batch_size, self.hidden_dim)
        forward_outputs = []
        for t in range(seq_len):
            h_forward = self.rnn_forward(X[t], h_forward)
            forward_outputs.append(h_forward)
        
        # 后向传播（从后往前）
        h_backward = torch.zeros(batch_size, self.hidden_dim)
        backward_outputs = []
        for t in range(seq_len - 1, -1, -1):
            h_backward = self.rnn_backward(X[t], h_backward)
            backward_outputs.insert(0, h_backward)  # 恢复到正序
        
        # 拼接前向和后向输出
        outputs = []
        for t in range(seq_len):
            combined = torch.cat([forward_outputs[t], backward_outputs[t]], dim=1)
            outputs.append(combined)
        
        outputs = torch.stack(outputs)  # (seq_len, batch, 2*hidden_dim)
        
        # 最终隐藏状态
        final_hidden = torch.cat([forward_outputs[-1], backward_outputs[-1]], dim=1)
        
        return outputs, final_hidden

# 测试
print("测试双向RNN编码器:")

seq_len = 5
batch_size = 3
input_dim = 8
hidden_dim = 6

X = torch.randn(seq_len, batch_size, input_dim)

# 使用PyTorch实现
encoder = BidirectionalRNNEncoder(input_dim, hidden_dim)
outputs, final_hidden = encoder(X)

print(f"输入形状: {X.shape}")
print(f"输出形状: {outputs.shape}")
print(f"最终隐藏状态形状: {final_hidden.shape}")

# 使用手动实现验证
encoder_manual = BidirectionalRNNManual(input_dim, hidden_dim)
outputs_manual, final_hidden_manual = encoder_manual(X)

print(f"\n手动实现输出形状: {outputs_manual.shape}")
print(f"手动实现最终隐藏状态形状: {final_hidden_manual.shape}")

# 验证维度一致性
print(f"\n维度验证:")
print(f"每个时间步输出维度: 2 * hidden_dim = {2 * hidden_dim}")
print(f"序列长度: {seq_len}")
print(f"批次大小: {batch_size}")

测试双向RNN编码器:
输入形状: torch.Size([5, 3, 8])
输出形状: torch.Size([5, 3, 12])
最终隐藏状态形状: torch.Size([3, 12])

手动实现输出形状: torch.Size([5, 3, 12])
手动实现最终隐藏状态形状: torch.Size([3, 12])

维度验证:
每个时间步输出维度: 2 * hidden_dim = 12
序列长度: 5
批次大小: 3


5 嵌入向量


5.1 Skip-gram负采样目标函数

给定中心词 w_c 和上下文词 w_o，负采样K个负样本。
 v_c: 中心词向量 (输入向量)
 u_o: 上下文词向量 (输出向量)
 u_{n_k}: 第k个负样本的词向量
 K: 负样本数量

损失函数（对数似然）:
L = -log σ(v_c · u_o) - Σ_{k=1}^{K} log σ(-v_c · u_{n_k})

其中 σ(x) = 1/(1 + exp(-x)) 是sigmoid函数

完整目标函数:
J = -[log σ(v_c · u_o) + Σ_{k=1}^{K} E_{n_k ~ P_n}[log σ(-v_c · u_{n_k})]]

负样本采样方法:
1. 从噪声分布 P_n(w) 中采样
2. 常用的噪声分布是 unigram 分布的 3/4 次方:
   P_n(w) = (count(w)^(3/4)) / Σ_j (count(j)^(3/4))
3. 这种分布偏向于采样高频词，但不会过度偏向


5.2 CBOW模型前向传播和损失计算

In [10]:
import torch
import torch.nn.functional as F

class CBOWModel:
    def __init__(self, vocab_size, embedding_dim):
        """
        CBOW模型
        Args:
            vocab_size: 词汇表大小 V
            embedding_dim: 嵌入维度 d
        """
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        
        # 初始化权重矩阵
        # 输入权重 W: (V, d)
        self.W = torch.randn(vocab_size, embedding_dim) * 0.01
        # 输出权重 W_out: (d, V)
        self.W_out = torch.randn(embedding_dim, vocab_size) * 0.01
        
    def forward(self, context_indices, target_idx):
        """
        前向传播和损失计算
        Args:
            context_indices: 上下文词索引列表 (batch_size, context_size)
            target_idx: 目标词索引 (batch_size,)
        Returns:
            loss: 交叉熵损失
            probs: 输出概率分布 (batch_size, vocab_size)
        """
        batch_size, context_size = context_indices.shape
        
        # 1. 获取上下文词的嵌入向量
        # context_embeds: (batch_size, context_size, embedding_dim)
        context_embeds = self.W[context_indices]
        
        # 2. 计算平均上下文向量作为隐藏层
        # hidden: (batch_size, embedding_dim)
        hidden = torch.mean(context_embeds, dim=1)
        
        # 3. 计算输出得分
        # scores: (batch_size, vocab_size)
        scores = torch.matmul(hidden, self.W_out)  # (batch_size, d) * (d, V) -> (batch_size, V)
        
        # 4. 计算输出概率分布（完整softmax）
        probs = F.softmax(scores, dim=1)  # (batch_size, V)
        
        # 5. 计算交叉熵损失
        # 使用负对数似然
        loss = F.cross_entropy(scores, target_idx)
        
        return loss, probs

# 测试CBOW模型
print("测试CBOW模型:")

vocab_size = 10
embedding_dim = 5
batch_size = 4
context_size = 3

# 创建模型
model = CBOWModel(vocab_size, embedding_dim)

# 生成随机数据
context_indices = torch.randint(0, vocab_size, (batch_size, context_size))
target_idx = torch.randint(0, vocab_size, (batch_size,))

print(f"词汇表大小 V = {vocab_size}")
print(f"嵌入维度 d = {embedding_dim}")
print(f"批次大小 = {batch_size}")
print(f"上下文大小 = {context_size}")
print(f"上下文索引形状: {context_indices.shape}")
print(f"目标索引形状: {target_idx.shape}")

# 前向传播
loss, probs = model.forward(context_indices, target_idx)

print(f"\n损失值: {loss.item():.4f}")
print(f"输出概率分布形状: {probs.shape}")
print(f"每个样本的概率之和: {probs.sum(dim=1)}")

# 查看第一个样本的上下文和目标
print(f"\n第一个样本:")
print(f"  上下文词索引: {context_indices[0].tolist()}")
print(f"  目标词索引: {target_idx[0].item()}")
print(f"  预测概率分布（前5个）: {probs[0, :5].tolist()}")
print(f"  目标词概率: {probs[0, target_idx[0]].item():.4f}")

测试CBOW模型:
词汇表大小 V = 10
嵌入维度 d = 5
批次大小 = 4
上下文大小 = 3
上下文索引形状: torch.Size([4, 3])
目标索引形状: torch.Size([4])

损失值: 2.3025
输出概率分布形状: torch.Size([4, 10])
每个样本的概率之和: tensor([1.0000, 1.0000, 1.0000, 1.0000])

第一个样本:
  上下文词索引: [1, 2, 1]
  目标词索引: 3
  预测概率分布（前5个）: [0.0999724417924881, 0.10001489520072937, 0.09999581426382065, 0.10000491887331009, 0.09998198598623276]
  目标词概率: 0.1000


6 注意力机制

6.1 缩放点积注意力计算
Q (2×4):
[[1 2 3 4]
 [5 6 7 8]]

K (3×4):
[[1 1 1 1]
 [2 2 2 2]
 [3 3 3 3]]

V (3×5):
[[ 1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 15]]

d_k = 4

1. 得分矩阵 S = Q·K^T / √d_k
   S (2×3):
[[ 5. 10. 15.]
 [13. 26. 39.]]

2. Softmax (按行归一化):
   Attention Weights A (2×3):
[[4.50940412e-05 6.69254912e-03 9.93262357e-01]
 [5.10907748e-12 2.26032430e-06 9.99997740e-01]]
   每行之和: [1. 1.]

3. 加权求和 Output = A·V
   Output (2×5):
[[10.96608631 11.96608631 12.96608631 13.96608631 14.96608631]
 [10.9999887  11.9999887  12.9999887  13.9999887  14.9999887 ]]


6.2 多头注意力的前向传播

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        """
        多头注意力
        Args:
            d_model: 模型维度
            num_heads: 注意力头数
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, X):
        """
        前向传播
        Args:
            X: 输入 (seq_len, batch, d_model)
        Returns:
            output: 输出 (seq_len, batch, d_model)
        """
        seq_len, batch_size, _ = X.shape
        
        # 1. 线性投影
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)  # (seq_len, batch, d_model)
        V = self.W_v(X)  # (seq_len, batch, d_model)
        
        # 2. 重塑为多头
        # (seq_len, batch, num_heads, d_k) -> (seq_len, batch*num_heads, d_k)
        Q = Q.view(seq_len, batch_size * self.num_heads, self.d_k)
        K = K.view(seq_len, batch_size * self.num_heads, self.d_k)
        V = V.view(seq_len, batch_size * self.num_heads, self.d_v)
        
        # 3. 缩放点积注意力
        # 转置Q和K以便计算
        Q = Q.transpose(0, 1)  # (batch*num_heads, seq_len, d_k)
        K = K.transpose(0, 1)  # (batch*num_heads, seq_len, d_k)
        V = V.transpose(0, 1)  # (batch*num_heads, seq_len, d_v)
        
        # 计算注意力得分
        scores = torch.bmm(Q, K.transpose(1, 2)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))
        
        # Softmax
        attention_weights = F.softmax(scores, dim=-1)
        
        # 加权求和
        head_outputs = torch.bmm(attention_weights, V)  # (batch*num_heads, seq_len, d_v)
        
        # 4. 重塑回原始形状
        # (batch*num_heads, seq_len, d_v) -> (seq_len, batch, num_heads, d_v)
        head_outputs = head_outputs.view(batch_size, self.num_heads, seq_len, self.d_v)
        head_outputs = head_outputs.transpose(0, 2)  # (seq_len, batch, num_heads, d_v)
        
        # 拼接所有头
        # (seq_len, batch, num_heads * d_v) = (seq_len, batch, d_model)
        concat_output = head_outputs.reshape(seq_len, batch_size, self.d_model)
        
        # 5. 最终线性层
        output = self.W_o(concat_output)
        
        return output

# 测试多头注意力
print("测试多头注意力:")

seq_len = 6
batch_size = 4
d_model = 8
num_heads = 2

# 创建模型
mha = MultiHeadAttention(d_model, num_heads)

# 生成输入
X = torch.randn(seq_len, batch_size, d_model)

print(f"序列长度: {seq_len}")
print(f"批次大小: {batch_size}")
print(f"模型维度 d_model: {d_model}")
print(f"注意力头数 num_heads: {num_heads}")
print(f"每个头的维度 d_k = d_v = d_model/num_heads = {d_model // num_heads}")

# 前向传播
output = mha.forward(X)

print(f"\n输入形状: {X.shape}")
print(f"输出形状: {output.shape}")
print(f"输入输出形状一致: {X.shape == output.shape}")

# 检查各头是否独立计算（可选）
print("\n前几个输出值:")
print(output[0, 0, :5].detach().numpy())

# 手动验证第一个头的维度
print(f"\n验证维度:")
print(f"每个头的维度: {d_model // num_heads}")
print(f"拼接后维度: {num_heads} * {d_model // num_heads} = {d_model}")

测试多头注意力:
序列长度: 6
批次大小: 4
模型维度 d_model: 8
注意力头数 num_heads: 2
每个头的维度 d_k = d_v = d_model/num_heads = 4

输入形状: torch.Size([6, 4, 8])
输出形状: torch.Size([6, 4, 8])
输入输出形状一致: True

前几个输出值:
[ 0.23795256 -0.18200572  0.0024739  -0.14270592 -0.11168005]

验证维度:
每个头的维度: 4
拼接后维度: 2 * 4 = 8
